In [ ]:
import pandas as pd
from catboost import CatBoostClassifier

TRAIN_PATH = "train.csv"
TEST_PATH  = "test_X.csv"
target_col = "is_fake"
id_col     = "id"

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

y = train[target_col].astype(int)
X = train.drop(columns=[target_col]).copy()
X_test = test.copy()

if id_col in X.columns:
    X.drop(columns=[id_col], inplace=True)
if id_col in X_test.columns:
    X_test.drop(columns=[id_col], inplace=True)

for c in ["seller_id", "brand", "category"]:
    if c in X.columns:
        X[c] = X[c].astype("string").fillna("NA")
        X_test[c] = X_test[c].astype("string").fillna("NA")

for c in ["title_name", "description"]:
    if c in X.columns:
        X[c] = X[c].astype("string").fillna("")
        X_test[c] = X_test[c].astype("string").fillna("")

count_cols = [
    "rating_1_count","rating_2_count","rating_3_count","rating_4_count","rating_5_count",
    "comments_count","photos_count","videos_count"
]
for c in count_cols:
    if c in X.columns:
        X[c] = pd.to_numeric(X[c], errors="coerce").fillna(0)
        X_test[c] = pd.to_numeric(X_test[c], errors="coerce").fillna(0)

if all(c in X.columns for c in ["rating_1_count","rating_2_count","rating_3_count","rating_4_count","rating_5_count"]):
    X["ratings_total"] = X[["rating_1_count","rating_2_count","rating_3_count","rating_4_count","rating_5_count"]].sum(axis=1)
    X_test["ratings_total"] = X_test[["rating_1_count","rating_2_count","rating_3_count","rating_4_count","rating_5_count"]].sum(axis=1)
    X["has_ratings"] = (X["ratings_total"] > 0).astype(int)
    X_test["has_ratings"] = (X_test["ratings_total"] > 0).astype(int)

if "comments_count" in X.columns:
    X["has_comments"] = (X["comments_count"] > 0).astype(int)
    X_test["has_comments"] = (X_test["comments_count"] > 0).astype(int)

if all(c in X.columns for c in ["photos_count","videos_count"]):
    X["media_total"] = X["photos_count"] + X["videos_count"]
    X_test["media_total"] = X_test["photos_count"] + X_test["videos_count"]
    X["has_media"] = (X["media_total"] > 0).astype(int)
    X_test["has_media"] = (X_test["media_total"] > 0).astype(int)

cat_features  = [c for c in ["brand", "category", "seller_id"] if c in X.columns]
text_features = [c for c in ["title_name", "description"] if c in X.columns]

FIXED_ITERS = 3115

model = CatBoostClassifier(
    iterations=FIXED_ITERS,
    learning_rate=0.12,
    depth=8,
    loss_function="Logloss",
    custom_loss="AUC",
    random_seed=42,
    auto_class_weights="Balanced",
    l2_leaf_reg=13.0,
    random_strength=1.5,
    verbose=200,
    thread_count=-1
)

model.fit(
    X, y,
    cat_features=cat_features,
    text_features=text_features
)

test_pred = model.predict_proba(X_test)[:, 1]
sub = pd.DataFrame({"id": test[id_col], "is_fake": test_pred})
sub.to_csv(f"submission_{FIXED_ITERS}.csv", index=False)

print(f"✅ saved submission_{FIXED_ITERS}.csv")